In [1]:
# bloc 1 - imports et configuration
import os
import joblib
import pandas as pd
import numpy as np
from typing import Tuple

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Chemin vers le pipeline minimal (préproc + RF)
MODEL_PATH = '/content/drive/MyDrive/data_P12/rf_pipeline_min.joblib'  # adapter si besoin

In [3]:
# bloc 2 - fonction de chargement
def load_rf_pipeline(model_path: str):

    """
    Charge un pipeline scikit-learn contenant le préprocesseur et RandomForest.
    Lève une erreur claire si le fichier est introuvable.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    model = joblib.load(model_path)
    return model


In [9]:
# bloc 3 - fonction principale
def predict_and_split(df: pd.DataFrame,
                      model,
                      target_col: str = None,
                      proba_col: str = 'is_genuine_proba_rf',
                      pred_col: str = 'is_genuine_pred_rf'
                      ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Retourne (df_all, df_true, df_false)
    - df_all : DataFrame original enrichi de proba_col et pred_col
    - df_true : lignes où pred_col == 1
    - df_false: lignes où pred_col == 0
    """
    df_all = df.copy()

    if not hasattr(model, 'predict_proba'):
        raise AttributeError("Le modèle fourni n'expose pas predict_proba. Fournissez un pipeline préproc+RF.")

    try:
        proba = model.predict_proba(df_all)[:, 1]
    except Exception as e:
        raise RuntimeError(f"Erreur lors de predict_proba. Vérifiez les colonnes/features. Détail: {e}")

    df_all[proba_col] = proba
    df_all[pred_col] = (df_all[proba_col] >= 0.5).astype(int)

    df_true = df_all[df_all[pred_col] == 1].copy()
    df_false = df_all[df_all[pred_col] == 0].copy()

    return df_all, df_true, df_false


In [5]:
# bloc 4 - exemple d'utilisation

SAMPLE_CSV = '/content/billets_production.csv'
df_sample = pd.read_csv(SAMPLE_CSV)
model = load_rf_pipeline(MODEL_PATH)
df_all, df_true, df_false = predict_and_split(df_sample, model)
print("Total:", len(df_all), "Vrais prédits:", len(df_true), "Faux prédits:", len(df_false))

Total: 5 Vrais prédits: 2 Faux prédits: 3
